## Gold Dimensions
Loads dimension tables of the YouTube trending star schema using idempotent MERGE (upsert).
Source: silver table `youtube_videos_ldp_silver`.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import Window

### dim_category
Static reference dimension: YouTube category code → name. One row per category.

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {gold['dim_category']} (category_id, category_name, _updated_at)
    SELECT category_id, category_name, current_timestamp()
    FROM VALUES
        (1,  'Film & Animation'), (2,  'Autos & Vehicles'), (10, 'Music'),
        (15, 'Pets & Animals'),   (17, 'Sports'),           (19, 'Travel & Events'),
        (20, 'Gaming'),           (22, 'People & Blogs'),   (23, 'Comedy'),
        (24, 'Entertainment'),    (25, 'News & Politics'),  (26, 'Howto & Style'),
        (27, 'Education'),        (28, 'Science & Technology'),
        (29, 'Nonprofits & Activism'), (43, 'Shows')
    AS v(category_id, category_name)
""")

### dim_channel


In [0]:
channel_src_df = (
    scoped_silver()
    .select("channel_title")
    .filter(F.col("channel_title").isNotNull() & (F.col("channel_title") != ""))
    .distinct()
    .withColumn("_updated_at", F.current_timestamp())
)

In [0]:
target = DeltaTable.forName(spark, gold["dim_channel"])
(
    target.alias("t")
    .merge(
        channel_src_df.alias("s"),
        "t.channel_title = s.channel_title"
    )
    .whenNotMatchedInsert(
        values={
            "channel_title": "s.channel_title",
            "_updated_at": "s._updated_at",
        }
    )
    .execute()
)

display(
    target.history(1)
    .select(
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
        F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
    )
)

### dim_video


In [0]:
latest_first = Window.partitionBy("video_id").orderBy(
    F.col("trending_dt").desc(),
    F.col("_silver_processed_at").desc(),
)

video_src_df = (
    scoped_silver()
    .filter(F.col("video_id").isNotNull())
    .withColumn("rn", F.row_number().over(latest_first))
    .filter(F.col("rn") == 1)
    .select(
        "video_id",
        "title",
        F.to_timestamp("publish_time").alias("publish_ts"),
        "tags",
        F.col("comments_disabled").cast("boolean").alias("comments_disabled"),
        F.col("ratings_disabled").cast("boolean").alias("ratings_disabled"),
    )
    .withColumn("_updated_at", F.current_timestamp())
)

In [0]:
target = DeltaTable.forName(spark, gold["dim_video"])

changed = """
    NOT (
        t.title             <=> s.title             AND
        t.publish_ts        <=> s.publish_ts        AND
        t.tags              <=> s.tags              AND
        t.comments_disabled <=> s.comments_disabled AND
        t.ratings_disabled  <=> s.ratings_disabled
    )
"""
(
    target.alias("t")
    .merge(
        video_src_df.alias("s"),
        "t.video_id = s.video_id"
    )
    .whenMatchedUpdate(
        condition = changed,
        set={
            "title": "s.title",
            "publish_ts": "s.publish_ts",
            "tags": "s.tags",
            "comments_disabled": "s.comments_disabled",
            "ratings_disabled": "s.ratings_disabled",
            "_updated_at": "s._updated_at",
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

display(
    target.history(1)
    .select(
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
        F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
    )
)

### dim_date
Calendar generated from the min to the max `trending_date` in silver.

In [0]:
date_src_df = (
    scoped_silver()
    .select(F.col("trending_dt").alias("d"))
    .agg(
        F.min("d").alias("start_date"),
        F.max("d").alias("end_date"),
    )
    .select(
        F.explode(
            F.sequence("start_date", "end_date", F.expr("INTERVAL 1 DAY"))
        ).alias("full_date")
    )
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("year"),
        F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day_of_month"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    )
    .withColumn("_updated_at", F.current_timestamp())
)

In [0]:
target = DeltaTable.forName(spark, gold["dim_date"])

(
    target.alias("t")
    .merge(
        date_src_df.alias("s"),
        "t.date_key = s.date_key"
    )
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

display(
    target.history(1)
    .select(
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
        F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
    )
)